In [ ]:
#Image transformation to 800*600
import cv2
import numpy as np
import os

def preprocess_image(img_path):
    img = cv2.imread(img_path)
    if img is None:
        raise ValueError(f"Image not found: {img_path}")

    h, w = img.shape[:2]
    target_aspect = 4 / 3

    # Crop to 4:3 aspect ratio (center crop)
    if w / h > target_aspect:
        new_w = int(h * target_aspect)
        crop_x = (w - new_w) // 2
        img = img[:, crop_x:crop_x + new_w]
    else:
        new_h = int(w / target_aspect)
        crop_y = (h - new_h) // 2
        img = img[crop_y:crop_y + new_h, :]

    # Resize only if image is larger than target
    h, w = img.shape[:2]
    if w > 800 or h > 600:
        img = cv2.resize(img, (800, 600), interpolation=cv2.INTER_AREA)
        return img

    # Pad smaller images to 800x600
    top = (600 - h) // 2
    bottom = 600 - h - top
    left = (800 - w) // 2
    right = 800 - w - left

    img = cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_CONSTANT, value=(0, 0, 0))
    return img


# -----------------------------
# Batch processing script
# -----------------------------
input_folder = r"F:\desktop\DS203\project\DS203-2025-S1-E7-project-images"
output_folder = os.path.join(os.path.dirname(input_folder), "images_transformed")

# Create output folder if not exists
os.makedirs(output_folder, exist_ok=True)

# Supported image extensions
valid_exts = (".jpg", ".jpeg", ".png", ".bmp", ".tiff")

for filename in os.listdir(input_folder):
    if filename.lower().endswith(valid_exts):
        input_path = os.path.join(input_folder, filename)
        try:
            processed = preprocess_image(input_path)
            output_path = os.path.join(output_folder, filename)
            cv2.imwrite(output_path, processed)
            print(f"✅ Saved: {output_path}")
        except Exception as e:
            print(f"❌ Error processing {filename}: {e}")

print("\n🎯 All images processed successfully!")
print(f"Transformed images saved in: {output_folder}")


In [ ]:
#HSV  GMM
import cv2
import numpy as np
import os
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt
# ----------------------------------------------
# Compute HSV color histogram + mean + std
# ----------------------------------------------
def compute_hsv_features(block):
    hsv = cv2.cvtColor(block, cv2.COLOR_BGR2HSV)

    # Color histogram (HSV)
    hist = cv2.calcHist([hsv], [0, 1, 2], None, [8, 8, 8], [0, 180, 0, 256, 0, 256])
    hist = cv2.normalize(hist, hist).flatten().astype(np.float64)

    # Mean and Std
    mean_vals = hsv.mean(axis=(0, 1))
    std_vals = hsv.std(axis=(0, 1))

    return np.concatenate([hist, mean_vals, std_vals]).astype(np.float64)

# ----------------------------------------------
# Extract block and sub-block features
# ----------------------------------------------
def extract_block_features(img, grid_size=(8, 8), sub_block=2):
    h, w = img.shape[:2]
    gh, gw = grid_size
    bh, bw = h // gh, w // gw
    features = []
    block_indices = []

    for i in range(gh):
        for j in range(gw):
            y0, y1 = i * bh, (i + 1) * bh
            x0, x1 = j * bw, (j + 1) * bw
            block = img[y0:y1, x0:x1]

            # Divide into sub-blocks
            sub_feats = []
            sb_h, sb_w = bh // sub_block, bw // sub_block
            for si in range(sub_block):
                for sj in range(sub_block):
                    sy0, sy1 = si * sb_h, (si + 1) * sb_h
                    sx0, sx1 = sj * sb_w, (sj + 1) * sb_w
                    sub_block_img = block[sy0:sy1, sx0:sx1]
                    sub_feats.append(compute_hsv_features(sub_block_img))

            features.extend(sub_feats)
            block_indices.append((i, j, len(sub_feats)))

    return np.array(features, dtype=np.float64), block_indices, (bh, bw)

# ----------------------------------------------
# Cluster blocks using GMM
# ----------------------------------------------
def cluster_blocks_gmm(features, n_clusters=2, pca_components=20, reg_covar=1e-3):
    scaler = StandardScaler()
    feats_scaled = scaler.fit_transform(features)

    pca = PCA(n_components=min(pca_components, feats_scaled.shape[1], feats_scaled.shape[0]-1))
    feats_pca = pca.fit_transform(feats_scaled)

    gmm = GaussianMixture(n_components=n_clusters, covariance_type='full',
                          random_state=42, reg_covar=reg_covar)
    gmm.fit(feats_pca)
    labels = gmm.predict(feats_pca)
    return labels

# ----------------------------------------------
# Aggregate sub-block labels to block labels
# ----------------------------------------------
def aggregate_majority(block_indices, sub_labels):
    gh = max(idx[0] for idx in block_indices) + 1
    gw = max(idx[1] for idx in block_indices) + 1
    block_labels = np.zeros((gh, gw), dtype=int)

    idx = 0
    for i, j, sub_count in block_indices:
        sub_block_labels = sub_labels[idx:idx + sub_count]
        counts = np.bincount(sub_block_labels)
        block_labels[i, j] = np.argmax(counts)
        idx += sub_count

    return block_labels

# ----------------------------------------------
# Visualize results (3 images per row)
# ----------------------------------------------
def visualize_results(image_results):
    n_images = len(image_results)
    cols = 3
    rows = (n_images + cols - 1) // cols

    plt.figure(figsize=(18, 6 * rows))
    for idx, res in enumerate(image_results):
        img = res['img']
        block_labels = res['labels']
        bh, bw = res['block_shape']
        gh, gw = block_labels.shape

        overlay = img.astype(np.float64) / 255.0
        red = np.array([1, 0, 0])
        for i in range(gh):
            for j in range(gw):
                if block_labels[i, j] == 1:  # highlight cluster 1
                    y0, y1 = i * bh, (i + 1) * bh
                    x0, x1 = j * bw, (j + 1) * bw
                    overlay[y0:y1, x0:x1] = 0.6 * overlay[y0:y1, x0:x1] + 0.4 * red
        result = (overlay * 255).astype(np.uint8)

        plt.subplot(rows, cols, idx + 1)
        plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
        plt.title(res['name'], fontsize=10)
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# ----------------------------------------------
# Full pipeline
# ----------------------------------------------
def process_folder_blocks(folder_path, n_images=15, sub_block=2):
    img_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
    img_files = sorted(img_files)[:n_images]
    results = []

    for img_file in img_files:
        img_path = os.path.join(folder_path, img_file)
        print(f"Processing: {img_file}")
        img = cv2.imread(img_path)

        features, block_indices, block_shape = extract_block_features(img, sub_block=sub_block)
        try:
            sub_labels = cluster_blocks_gmm(features, n_clusters=2, reg_covar=1e-3)
        except Exception as e:
            print(f"GMM failed on {img_file}: {e}")
            continue

        block_labels = aggregate_majority(block_indices, sub_labels)
        results.append({'name': img_file, 'img': img, 'labels': block_labels, 'block_shape': block_shape})

    visualize_results(results)
    return results

# ----------------------------------------------
# Run pipeline
# ----------------------------------------------
folder_path = r"F:\desktop\DS203\project\images_transformed"
results = process_folder_blocks(folder_path, n_images=15, sub_block=2)


In [ ]:
#Edge Detection Using Sobel Canny Laplacian
# ----------------------------------------------
# Edge Detection Functions
# ----------------------------------------------
def detect_edges(gray, sobel_thresh=50, laplacian_thresh=40):
    # --- Canny Edge ---
    edges_canny = cv2.Canny(gray, 100, 200)

    # --- Sobel Edge ---
    sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    sobel_mag = cv2.magnitude(sobelx, sobely)
    sobel_mag = cv2.convertScaleAbs(sobel_mag)
    _, edges_sobel = cv2.threshold(sobel_mag, sobel_thresh, 255, cv2.THRESH_BINARY)

    # --- Laplacian Edge ---
    laplacian = cv2.Laplacian(gray, cv2.CV_64F, ksize=3)
    laplacian = cv2.convertScaleAbs(laplacian)
    _, edges_laplacian = cv2.threshold(laplacian, laplacian_thresh, 255, cv2.THRESH_BINARY)

    return edges_canny, edges_sobel, edges_laplacian

# ----------------------------------------------
# Process folder and visualize all edges
# ----------------------------------------------
def process_edge_detection(folder_path, n_images=15):
    img_files = [f for f in os.listdir(folder_path)
                 if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
    img_files = sorted(img_files)[:n_images]

    n = len(img_files)
    plt.figure(figsize=(15, 5 * n))  # 3 columns per image

    for idx, img_file in enumerate(img_files):
        img_path = os.path.join(folder_path, img_file)
        print(f"Processing: {img_file}")
        img = cv2.imread(img_path)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # Detect edges
        canny, sobel, laplacian = detect_edges(gray)

        # Plot results side by side
        plt.subplot(n, 3, 3*idx + 1)
        plt.imshow(canny, cmap='gray')
        plt.title(f"{img_file} - Canny", fontsize=8)
        plt.axis('off')

        plt.subplot(n, 3, 3*idx + 2)
        plt.imshow(sobel, cmap='gray')
        plt.title("Sobel", fontsize=8)
        plt.axis('off')

        plt.subplot(n, 3, 3*idx + 3)
        plt.imshow(laplacian, cmap='gray')
        plt.title("Laplacian", fontsize=8)
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# ----------------------------------------------
# Run the pipeline
# ----------------------------------------------
folder_path = r"F:\desktop\DS203\project\images_transformed"
process_edge_detection(folder_path, n_images=15)


In [ ]:
#HSV Laplacian Kmeans
from sklearn.cluster import KMeans

# ----------------------------------------------
# Compute HSV + Laplacian edge features
# ----------------------------------------------
def compute_features_with_laplacian(block):
    # --- HSV Features ---
    hsv = cv2.cvtColor(block, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0,1,2], None, [8,8,8], [0,180,0,256,0,256])
    hist = cv2.normalize(hist, hist).flatten().astype(np.float64)
    mean_vals = hsv.mean(axis=(0,1))
    std_vals = hsv.std(axis=(0,1))
    hsv_features = np.concatenate([hist, mean_vals, std_vals])

    # --- Laplacian Edge Features ---
    gray = cv2.cvtColor(block, cv2.COLOR_BGR2GRAY)
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    lap_density = np.sum(np.abs(lap) > 20) / lap.size
    lap_mean, lap_std = lap.mean(), lap.std()
    edge_features = np.array([lap_density, lap_mean, lap_std], dtype=np.float64)

    return hsv_features, edge_features

# ----------------------------------------------
# Extract block and sub-block features
# ----------------------------------------------
def extract_block_features(img, grid_size=(8,8), sub_block=2):
    h, w = img.shape[:2]
    gh, gw = grid_size
    bh, bw = h // gh, w // gw
    hsv_features_list = []
    edge_features_list = []
    block_indices = []

    for i in range(gh):
        for j in range(gw):
            y0, y1 = i*bh, (i+1)*bh
            x0, x1 = j*bw, (j+1)*bw
            block = img[y0:y1, x0:x1]

            sub_hsv_feats = []
            sub_edge_feats = []

            sb_h, sb_w = bh // sub_block, bw // sub_block
            for si in range(sub_block):
                for sj in range(sub_block):
                    sy0, sy1 = si*sb_h, (si+1)*sb_h
                    sx0, sx1 = sj*sb_w, (sj+1)*sb_w
                    sub_block_img = block[sy0:sy1, sx0:sx1]
                    hsv_f, edge_f = compute_features_with_laplacian(sub_block_img)
                    sub_hsv_feats.append(hsv_f)
                    sub_edge_feats.append(edge_f)

            hsv_features_list.extend(sub_hsv_feats)
            edge_features_list.extend(sub_edge_feats)
            block_indices.append((i, j, len(sub_hsv_feats)))

    return np.array(hsv_features_list), np.array(edge_features_list), block_indices, (bh, bw)

# ----------------------------------------------
# Combine features with scaling and edge weighting
# ----------------------------------------------
def combine_features(hsv_feats, edge_feats, edge_weight=3.0):
    # Scale separately
    hsv_scaled = StandardScaler().fit_transform(hsv_feats)
    edge_scaled = StandardScaler().fit_transform(edge_feats)
    # Apply weight to edge features
    edge_scaled *= edge_weight
    # Combine
    combined = np.hstack([hsv_scaled, edge_scaled])
    return combined

# ----------------------------------------------
# Cluster sub-blocks using KMeans + PCA
# ----------------------------------------------
def cluster_blocks_kmeans(features, n_clusters=2, pca_components=20):
    # PCA
    pca = PCA(n_components=min(pca_components, features.shape[1], features.shape[0]-1))
    feats_pca = pca.fit_transform(features)

    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    kmeans.fit(feats_pca)
    return kmeans.labels_

# ----------------------------------------------
# Aggregate sub-block labels to block labels
# ----------------------------------------------
def aggregate_majority(block_indices, sub_labels):
    gh = max(idx[0] for idx in block_indices)+1
    gw = max(idx[1] for idx in block_indices)+1
    block_labels = np.zeros((gh, gw), dtype=int)

    idx = 0
    for i,j,sub_count in block_indices:
        sub_block_labels = sub_labels[idx:idx+sub_count]
        counts = np.bincount(sub_block_labels)
        block_labels[i,j] = np.argmax(counts)
        idx += sub_count

    return block_labels

# ----------------------------------------------
# Visualize results (3 images per row)
# ----------------------------------------------
def visualize_results(image_results):
    n_images = len(image_results)
    cols = 3
    rows = (n_images + cols - 1)//cols
    plt.figure(figsize=(18, 6*rows))

    for idx, res in enumerate(image_results):
        img = res['img']
        block_labels = res['labels']
        bh, bw = res['block_shape']
        gh, gw = block_labels.shape

        overlay = img.astype(np.float64)/255.0
        red = np.array([1,0,0])
        for i in range(gh):
            for j in range(gw):
                if block_labels[i,j] == 1:
                    y0, y1 = i*bh, (i+1)*bh
                    x0, x1 = j*bw, (j+1)*bw
                    overlay[y0:y1, x0:x1] = 0.6*overlay[y0:y1,x0:x1] + 0.4*red
        result = (overlay*255).astype(np.uint8)

        plt.subplot(rows, cols, idx+1)
        plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
        plt.title(res['name'], fontsize=10)
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# ----------------------------------------------
# Full pipeline
# ----------------------------------------------
def process_folder_blocks_kmeans(folder_path, n_images=15, sub_block=4, edge_weight=3.0):
    img_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg','.png','.jpeg'))]
    img_files = sorted(img_files)[:n_images]
    results = []

    for img_file in img_files:
        img_path = os.path.join(folder_path, img_file)
        print(f"Processing: {img_file}")
        img = cv2.imread(img_path)

        hsv_feats, edge_feats, block_indices, block_shape = extract_block_features(img, sub_block=sub_block)
        combined_feats = combine_features(hsv_feats, edge_feats, edge_weight=edge_weight)

        try:
            sub_labels = cluster_blocks_kmeans(combined_feats, n_clusters=2, pca_components=20)
        except Exception as e:
            print(f"KMeans failed on {img_file}: {e}")
            continue

        block_labels = aggregate_majority(block_indices, sub_labels)
        results.append({'name': img_file, 'img': img, 'labels': block_labels, 'block_shape': block_shape})

    visualize_results(results)
    return results

# ----------------------------------------------
# Run pipeline
# ----------------------------------------------
folder_path = r"F:\desktop\DS203\project\images_transformed"
results = process_folder_blocks_kmeans(folder_path, n_images=15, sub_block=4, edge_weight=1.5)


In [ ]:
#Laplacian Kmeans
# ----------------------------------------------
# Compute Laplacian edge features only
# ----------------------------------------------
def compute_edge_features(block):
    gray = cv2.cvtColor(block, cv2.COLOR_BGR2GRAY)
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    lap_density = np.sum(np.abs(lap) > 20) / lap.size
    lap_mean, lap_std = lap.mean(), lap.std()
    return np.array([lap_density, lap_mean, lap_std], dtype=np.float64)

# ----------------------------------------------
# Extract block and sub-block edge features
# ----------------------------------------------
def extract_block_edge_features(img, grid_size=(8,8), sub_block=2):
    h, w = img.shape[:2]
    gh, gw = grid_size
    bh, bw = h // gh, w // gw
    edge_features_list = []
    block_indices = []

    for i in range(gh):
        for j in range(gw):
            y0, y1 = i*bh, (i+1)*bh
            x0, x1 = j*bw, (j+1)*bw
            block = img[y0:y1, x0:x1]

            sub_edge_feats = []
            sb_h, sb_w = bh // sub_block, bw // sub_block
            for si in range(sub_block):
                for sj in range(sub_block):
                    sy0, sy1 = si*sb_h, (si+1)*sb_h
                    sx0, sx1 = sj*sb_w, (sj+1)*sb_w
                    sub_block_img = block[sy0:sy1, sx0:sx1]
                    sub_edge_feats.append(compute_edge_features(sub_block_img))

            edge_features_list.extend(sub_edge_feats)
            block_indices.append((i, j, len(sub_edge_feats)))

    return np.array(edge_features_list), block_indices, (bh, bw)

# ----------------------------------------------
# Combine features (scale only)
# ----------------------------------------------
def scale_features(edge_feats):
    return StandardScaler().fit_transform(edge_feats)

# ----------------------------------------------
# Cluster sub-blocks using KMeans + PCA
# ----------------------------------------------
def cluster_blocks_kmeans(edge_feats, n_clusters=2, pca_components=3):
    # PCA
    pca = PCA(n_components=min(pca_components, edge_feats.shape[1], edge_feats.shape[0]-1))
    feats_pca = pca.fit_transform(edge_feats)

    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    kmeans.fit(feats_pca)
    return kmeans.labels_

# ----------------------------------------------
# Aggregate sub-block labels to block labels
# ----------------------------------------------
def aggregate_majority(block_indices, sub_labels):
    gh = max(idx[0] for idx in block_indices)+1
    gw = max(idx[1] for idx in block_indices)+1
    block_labels = np.zeros((gh, gw), dtype=int)

    idx = 0
    for i,j,sub_count in block_indices:
        sub_block_labels = sub_labels[idx:idx+sub_count]
        counts = np.bincount(sub_block_labels)
        block_labels[i,j] = np.argmax(counts)
        idx += sub_count

    return block_labels

# ----------------------------------------------
# Visualize results (3 images per row)
# ----------------------------------------------
def visualize_results(image_results):
    n_images = len(image_results)
    cols = 3
    rows = (n_images + cols - 1)//cols
    plt.figure(figsize=(18, 6*rows))

    for idx, res in enumerate(image_results):
        img = res['img']
        block_labels = res['labels']
        bh, bw = res['block_shape']
        gh, gw = block_labels.shape

        overlay = img.astype(np.float64)/255.0
        red = np.array([1,0,0])
        for i in range(gh):
            for j in range(gw):
                if block_labels[i,j] == 1:
                    y0, y1 = i*bh, (i+1)*bh
                    x0, x1 = j*bw, (j+1)*bw
                    overlay[y0:y1, x0:x1] = 0.6*overlay[y0:y1,x0:x1] + 0.4*red
        result = (overlay*255).astype(np.uint8)

        plt.subplot(rows, cols, idx+1)
        plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
        plt.title(res['name'], fontsize=10)
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# ----------------------------------------------
# Full pipeline
# ----------------------------------------------
def process_folder_blocks_edges(folder_path, n_images=15, sub_block=4):
    img_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg','.png','.jpeg'))]
    img_files = sorted(img_files)[:n_images]
    results = []

    for img_file in img_files:
        img_path = os.path.join(folder_path, img_file)
        print(f"Processing: {img_file}")
        img = cv2.imread(img_path)

        edge_feats, block_indices, block_shape = extract_block_edge_features(img, sub_block=sub_block)
        edge_feats_scaled = scale_features(edge_feats)

        try:
            sub_labels = cluster_blocks_kmeans(edge_feats_scaled, n_clusters=2, pca_components=3)
        except Exception as e:
            print(f"KMeans failed on {img_file}: {e}")
            continue

        block_labels = aggregate_majority(block_indices, sub_labels)
        results.append({'name': img_file, 'img': img, 'labels': block_labels, 'block_shape': block_shape})

    visualize_results(results)
    return results

# ----------------------------------------------
# Run pipeline
# ----------------------------------------------
folder_path = r"F:\desktop\DS203\project\images_transformed"
results = process_folder_blocks_edges(folder_path, n_images=15, sub_block=4)


In [ ]:
#Saliency Heatmap

# -------------------------------
# Compute saliency map
# -------------------------------
def compute_saliency(img):
    saliency = cv2.saliency.StaticSaliencyFineGrained_create()
    (success, saliency_map) = saliency.computeSaliency(img)
    saliency_map = (saliency_map * 255).astype(np.uint8)
    return saliency_map

# -------------------------------
# Visualize saliency maps (3 per row)
# -------------------------------
def visualize_saliency_maps(folder_path, n_images=15):
    img_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
    img_files = sorted(img_files)[:n_images]

    cols = 3
    rows = (n_images + cols - 1) // cols
    plt.figure(figsize=(18, 6 * rows))

    for idx, img_file in enumerate(img_files):
        img_path = os.path.join(folder_path, img_file)
        img = cv2.imread(img_path)
        sal_map = compute_saliency(img)

        plt.subplot(rows, cols, idx + 1)
        plt.imshow(sal_map, cmap='hot')
        plt.title(img_file, fontsize=10)
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# -------------------------------
# Run on folder
# -------------------------------
folder_path = r"F:\desktop\DS203\project\images_transformed"
visualize_saliency_maps(folder_path, n_images=15)


In [ ]:
#Saliency Kmeans
# ----------------------------------------------
# Compute saliency map
# ----------------------------------------------
def compute_saliency(img):
    # Simple spectral residual approach in OpenCV
    saliency_detector = cv2.saliency.StaticSaliencySpectralResidual_create()
    success, sal_map = saliency_detector.computeSaliency(img)
    sal_map = (sal_map * 255).astype(np.uint8)
    return sal_map

# ----------------------------------------------
# Compute saliency features for a sub-block
# ----------------------------------------------
def compute_saliency_features(sub_block_img, sal_map_sub):
    # Use the corresponding sub-block in the saliency map
    saliency_avg = np.mean(sal_map_sub)
    saliency_std = np.std(sal_map_sub)
    return np.array([saliency_avg, saliency_std], dtype=np.float64)

# ----------------------------------------------
# Extract sub-block saliency features
# ----------------------------------------------
def extract_block_saliency_features(img, grid_size=(8,8), sub_block=2):
    h, w = img.shape[:2]
    gh, gw = grid_size
    bh, bw = h // gh, w // gw
    features_list = []
    block_indices = []

    # Compute saliency map once
    sal_map = compute_saliency(img)

    for i in range(gh):
        for j in range(gw):
            y0, y1 = i*bh, (i+1)*bh
            x0, x1 = j*bw, (j+1)*bw
            block = img[y0:y1, x0:x1]
            sal_block = sal_map[y0:y1, x0:x1]

            sub_feats = []
            sb_h, sb_w = bh // sub_block, bw // sub_block
            for si in range(sub_block):
                for sj in range(sub_block):
                    sy0, sy1 = si*sb_h, (si+1)*sb_h
                    sx0, sx1 = sj*sb_w, (sj+1)*sb_w
                    sub_block_img = block[sy0:sy1, sx0:sx1]
                    sub_sal_map = sal_block[sy0:sy1, sx0:sx1]
                    sub_feats.append(compute_saliency_features(sub_block_img, sub_sal_map))

            features_list.extend(sub_feats)
            block_indices.append((i,j,len(sub_feats)))

    return np.array(features_list, dtype=np.float64), block_indices, (bh, bw)

# ----------------------------------------------
# Scale features
# ----------------------------------------------
def scale_features(features):
    return StandardScaler().fit_transform(features)

# ----------------------------------------------
# Cluster sub-blocks using KMeans + PCA
# ----------------------------------------------
def cluster_blocks_kmeans(features, n_clusters=2, pca_components=2):
    # PCA
    pca = PCA(n_components=min(pca_components, features.shape[1], features.shape[0]-1))
    feats_pca = pca.fit_transform(features)

    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    kmeans.fit(feats_pca)
    return kmeans.labels_

# ----------------------------------------------
# Aggregate sub-block labels to block labels
# ----------------------------------------------
def aggregate_majority(block_indices, sub_labels):
    gh = max(idx[0] for idx in block_indices)+1
    gw = max(idx[1] for idx in block_indices)+1
    block_labels = np.zeros((gh, gw), dtype=int)

    idx = 0
    for i,j,sub_count in block_indices:
        sub_block_labels = sub_labels[idx:idx+sub_count]
        counts = np.bincount(sub_block_labels)
        block_labels[i,j] = np.argmax(counts)
        idx += sub_count

    return block_labels

# ----------------------------------------------
# Visualize results (3 images per row)
# ----------------------------------------------
def visualize_results(image_results):
    n_images = len(image_results)
    cols = 3
    rows = (n_images + cols - 1)//cols
    plt.figure(figsize=(18, 6*rows))

    for idx, res in enumerate(image_results):
        img = res['img']
        block_labels = res['labels']
        bh, bw = res['block_shape']
        gh, gw = block_labels.shape

        overlay = img.astype(np.float64)/255.0
        red = np.array([1,0,0])
        for i in range(gh):
            for j in range(gw):
                if block_labels[i,j] == 1:
                    y0, y1 = i*bh, (i+1)*bh
                    x0, x1 = j*bw, (j+1)*bw
                    overlay[y0:y1, x0:x1] = 0.6*overlay[y0:y1,x0:x1] + 0.4*red
        result = (overlay*255).astype(np.uint8)

        plt.subplot(rows, cols, idx+1)
        plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
        plt.title(res['name'], fontsize=10)
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# ----------------------------------------------
# Full pipeline
# ----------------------------------------------
def process_folder_blocks_saliency(folder_path, n_images=15, sub_block=4):
    img_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg','.png','.jpeg'))]
    img_files = sorted(img_files)[:n_images]
    results = []

    for img_file in img_files:
        img_path = os.path.join(folder_path, img_file)
        print(f"Processing: {img_file}")
        img = cv2.imread(img_path)

        features, block_indices, block_shape = extract_block_saliency_features(img, sub_block=sub_block)
        features_scaled = scale_features(features)

        try:
            sub_labels = cluster_blocks_kmeans(features_scaled, n_clusters=2, pca_components=2)
        except Exception as e:
            print(f"KMeans failed on {img_file}: {e}")
            continue

        block_labels = aggregate_majority(block_indices, sub_labels)
        results.append({'name': img_file, 'img': img, 'labels': block_labels, 'block_shape': block_shape})

    visualize_results(results)
    return results

# ----------------------------------------------
# Run pipeline
# ----------------------------------------------
folder_path = r"F:\desktop\DS203\project\images_transformed"
results = process_folder_blocks_saliency(folder_path, n_images=15, sub_block=4)


In [ ]:
#HSV Saliency  Laplacian Kmeans

# ----------------------------------------------
# Compute saliency map
# ----------------------------------------------
def compute_saliency(img):
    saliency_detector = cv2.saliency.StaticSaliencySpectralResidual_create()
    success, sal_map = saliency_detector.computeSaliency(img)
    sal_map = (sal_map * 255).astype(np.uint8)
    return sal_map

# ----------------------------------------------
# Compute features for a sub-block
# ----------------------------------------------
def compute_combined_features(sub_block_img, sal_map_sub):
    # HSV mean and std
    hsv = cv2.cvtColor(sub_block_img, cv2.COLOR_BGR2HSV)
    hsv_mean = hsv.mean(axis=(0,1))
    hsv_std = hsv.std(axis=(0,1))

    # Laplacian edge features
    gray = cv2.cvtColor(sub_block_img, cv2.COLOR_BGR2GRAY)
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    lap_density = np.sum(np.abs(lap) > 20) / lap.size
    lap_mean = lap.mean()
    lap_std = lap.std()

    # Saliency features
    saliency_avg = np.mean(sal_map_sub)
    saliency_std = np.std(sal_map_sub)

    # Combine all
    return np.concatenate([
        hsv_mean, hsv_std,
        [lap_density, lap_mean, lap_std],
        [saliency_avg, saliency_std]
    ]).astype(np.float64)

# ----------------------------------------------
# Extract block and sub-block features
# ----------------------------------------------
def extract_block_combined_features(img, grid_size=(8,8), sub_block=2):
    h, w = img.shape[:2]
    gh, gw = grid_size
    bh, bw = h // gh, w // gw
    features_list = []
    block_indices = []

    sal_map = compute_saliency(img)

    for i in range(gh):
        for j in range(gw):
            y0, y1 = i*bh, (i+1)*bh
            x0, x1 = j*bw, (j+1)*bw
            block = img[y0:y1, x0:x1]
            sal_block = sal_map[y0:y1, x0:x1]

            sub_feats = []
            sb_h, sb_w = bh // sub_block, bw // sub_block
            for si in range(sub_block):
                for sj in range(sub_block):
                    sy0, sy1 = si*sb_h, (si+1)*sb_h
                    sx0, sx1 = sj*sb_w, (sj+1)*sb_w
                    sub_block_img = block[sy0:sy1, sx0:sx1]
                    sub_sal_map = sal_block[sy0:sy1, sx0:sx1]
                    sub_feats.append(compute_combined_features(sub_block_img, sub_sal_map))

            features_list.extend(sub_feats)
            block_indices.append((i,j,len(sub_feats)))

    return np.array(features_list, dtype=np.float64), block_indices, (bh, bw)

# ----------------------------------------------
# Scale features
# ----------------------------------------------
def scale_features(features):
    return StandardScaler().fit_transform(features)

# ----------------------------------------------
# Cluster sub-blocks using KMeans + PCA
# ----------------------------------------------
def cluster_blocks_kmeans(features, n_clusters=2, pca_components=5):
    pca = PCA(n_components=min(pca_components, features.shape[1], features.shape[0]-1))
    feats_pca = pca.fit_transform(features)

    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    kmeans.fit(feats_pca)
    return kmeans.labels_

# ----------------------------------------------
# Aggregate sub-block labels to block labels
# ----------------------------------------------
def aggregate_majority(block_indices, sub_labels):
    gh = max(idx[0] for idx in block_indices)+1
    gw = max(idx[1] for idx in block_indices)+1
    block_labels = np.zeros((gh, gw), dtype=int)

    idx = 0
    for i,j,sub_count in block_indices:
        sub_block_labels = sub_labels[idx:idx+sub_count]
        counts = np.bincount(sub_block_labels)
        block_labels[i,j] = np.argmax(counts)
        idx += sub_count

    return block_labels

# ----------------------------------------------
# Visualize results (3 images per row)
# ----------------------------------------------
def visualize_results(image_results):
    n_images = len(image_results)
    cols = 3
    rows = (n_images + cols - 1)//cols
    plt.figure(figsize=(18, 6*rows))

    for idx, res in enumerate(image_results):
        img = res['img']
        block_labels = res['labels']
        bh, bw = res['block_shape']
        gh, gw = block_labels.shape

        overlay = img.astype(np.float64)/255.0
        red = np.array([1,0,0])
        for i in range(gh):
            for j in range(gw):
                if block_labels[i,j] == 1:
                    y0, y1 = i*bh, (i+1)*bh
                    x0, x1 = j*bw, (j+1)*bw
                    overlay[y0:y1, x0:x1] = 0.6*overlay[y0:y1,x0:x1] + 0.4*red
        result = (overlay*255).astype(np.uint8)

        plt.subplot(rows, cols, idx+1)
        plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
        plt.title(res['name'], fontsize=10)
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# ----------------------------------------------
# Full pipeline
# ----------------------------------------------
def process_folder_blocks_combined(folder_path, n_images=15, sub_block=4):
    img_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg','.png','.jpeg'))]
    img_files = sorted(img_files)[:n_images]
    results = []

    for img_file in img_files:
        img_path = os.path.join(folder_path, img_file)
        print(f"Processing: {img_file}")
        img = cv2.imread(img_path)

        features, block_indices, block_shape = extract_block_combined_features(img, sub_block=sub_block)
        features_scaled = scale_features(features)

        try:
            sub_labels = cluster_blocks_kmeans(features_scaled, n_clusters=2, pca_components=5)
        except Exception as e:
            print(f"KMeans failed on {img_file}: {e}")
            continue

        block_labels = aggregate_majority(block_indices, sub_labels)
        results.append({'name': img_file, 'img': img, 'labels': block_labels, 'block_shape': block_shape})

    visualize_results(results)
    return results

# ----------------------------------------------
# Run pipeline
# ----------------------------------------------
folder_path = r"F:\desktop\DS203\project\images_transformed"
results = process_folder_blocks_combined(folder_path, n_images=15, sub_block=4)


In [ ]:
# Saliency HSV Laplacian LBP Kmeans
from skimage.feature import local_binary_pattern
# ----------------------------------------------
# Compute combined features for a block
# ----------------------------------------------
def compute_block_features(block, weights={'saliency':1.0, 'color':1.0, 'edge':1.0, 'texture':1.0}):
    # --- Saliency ---
    saliency_obj = cv2.saliency.StaticSaliencySpectralResidual_create()
    (success, saliencyMap) = saliency_obj.computeSaliency(block)
    saliencyMap = (saliencyMap*255).astype(np.float64)
    sal_avg, sal_std = saliencyMap.mean(), saliencyMap.std()
    sal_feat = np.array([sal_avg, sal_std]) * weights['saliency']

    # --- Color (HSV mean + std) ---
    hsv = cv2.cvtColor(block, cv2.COLOR_BGR2HSV).astype(np.float64)
    hsv_mean = hsv.mean(axis=(0,1))
    hsv_std = hsv.std(axis=(0,1))
    color_feat = np.concatenate([hsv_mean, hsv_std]) * weights['color']

    # --- Edge (Laplacian) ---
    gray = cv2.cvtColor(block, cv2.COLOR_BGR2GRAY)
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    edge_density = np.sum(np.abs(lap)>20)/lap.size
    edge_mean, edge_std = lap.mean(), lap.std()
    edge_feat = np.array([edge_density, edge_mean, edge_std]) * weights['edge']

    # --- Texture (LBP histogram) ---
    lbp = local_binary_pattern(gray, P=8, R=1, method='uniform')
    lbp_hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0,11), range=(0,10))
    lbp_hist = lbp_hist.astype(np.float64)
    lbp_hist /= (lbp_hist.sum() + 1e-6)
    texture_feat = lbp_hist * weights['texture']

    # Combine all
    features = np.concatenate([sal_feat, color_feat, edge_feat, texture_feat])
    return features

# ----------------------------------------------
# Extract features for sub-blocks
# ----------------------------------------------
def extract_features(img, grid_size=(8,8), sub_block=2, weights=None):
    if weights is None:
        weights = {'saliency':1.0, 'color':1.0, 'edge':1.0, 'texture':1.0}

    h, w = img.shape[:2]
    gh, gw = grid_size
    bh, bw = h//gh, w//gw

    features = []
    block_indices = []

    for i in range(gh):
        for j in range(gw):
            y0, y1 = i*bh, (i+1)*bh
            x0, x1 = j*bw, (j+1)*bw
            block = img[y0:y1, x0:x1]

            sub_feats = []
            sb_h, sb_w = bh//sub_block, bw//sub_block
            for si in range(sub_block):
                for sj in range(sub_block):
                    sy0, sy1 = si*sb_h, (si+1)*sb_h
                    sx0, sx1 = sj*sb_w, (sj+1)*sb_w
                    sub_block_img = block[sy0:sy1, sx0:sx1]
                    sub_feats.append(compute_block_features(sub_block_img, weights))

            features.extend(sub_feats)
            block_indices.append((i,j,len(sub_feats)))

    return np.array(features, dtype=np.float64), block_indices, (bh,bw)

# ----------------------------------------------
# Scale + PCA + KMeans clustering
# ----------------------------------------------
def cluster_features(features, n_clusters=2, pca_components=20):
    scaler = StandardScaler()
    feats_scaled = scaler.fit_transform(features)
    pca = PCA(n_components=min(pca_components, feats_scaled.shape[1], feats_scaled.shape[0]-1))
    feats_pca = pca.fit_transform(feats_scaled)
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    kmeans.fit(feats_pca)
    labels = kmeans.labels_
    return labels

# ----------------------------------------------
# Aggregate sub-block labels to block
# ----------------------------------------------
def aggregate_majority(block_indices, sub_labels):
    gh = max(idx[0] for idx in block_indices)+1
    gw = max(idx[1] for idx in block_indices)+1
    block_labels = np.zeros((gh,gw),dtype=int)
    idx=0
    for i,j,sub_count in block_indices:
        sub_block_labels = sub_labels[idx:idx+sub_count]
        counts = np.bincount(sub_block_labels)
        block_labels[i,j] = np.argmax(counts)
        idx += sub_count
    return block_labels

# ----------------------------------------------
# Visualization
# ----------------------------------------------
def visualize_blocks(img_list, labels_list, block_shapes, cols=3):
    n_images = len(img_list)
    rows = (n_images + cols - 1)//cols
    plt.figure(figsize=(18,6*rows))
    for idx in range(n_images):
        img = img_list[idx].copy().astype(np.float64)/255.0
        labels = labels_list[idx]
        bh, bw = block_shapes[idx]
        gh, gw = labels.shape
        overlay = img.copy()
        red = np.array([1,0,0])
        for i in range(gh):
            for j in range(gw):
                if labels[i,j]==1:
                    y0,y1 = i*bh,(i+1)*bh
                    x0,x1 = j*bw,(j+1)*bw
                    overlay[y0:y1,x0:x1] = 0.6*overlay[y0:y1,x0:x1]+0.4*red
        plt.subplot(rows, cols, idx+1)
        plt.imshow(cv2.cvtColor((overlay*255).astype(np.uint8), cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.title(f"Image {idx+1}")
    plt.tight_layout()
    plt.show()

# ----------------------------------------------
# Full pipeline
# ----------------------------------------------
def process_folder_combined_features(folder_path, n_images=15, sub_block=2, weights=None):
    img_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg','.png','.jpeg'))]
    img_files = sorted(img_files)[:n_images]
    img_list, labels_list, block_shapes = [], [], []

    for img_file in img_files:
        img_path = os.path.join(folder_path, img_file)
        print(f"Processing {img_file}")
        img = cv2.imread(img_path)
        features, block_indices, block_shape = extract_features(img, sub_block=sub_block, weights=weights)
        sub_labels = cluster_features(features, n_clusters=2)
        block_labels = aggregate_majority(block_indices, sub_labels)
        img_list.append(img)
        labels_list.append(block_labels)
        block_shapes.append(block_shape)

    visualize_blocks(img_list, labels_list, block_shapes)
    return img_list, labels_list, block_shapes

# ----------------------------------------------
# Example usage
# ----------------------------------------------
folder_path = r"F:\desktop\DS203\project\images_transformed"
weights = {'saliency':2.0, 'color':1.0, 'edge':2.0, 'texture':1.0}  # edge & saliency are weighted more
images, labels, shapes = process_folder_combined_features(folder_path, n_images=15, sub_block=4, weights=weights)


In [ ]:
import cv2
import numpy as np
import os
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from skimage.feature import local_binary_pattern
from sklearn.mixture import GaussianMixture
# ----------------------------------------------
# Compute combined features for a block
# ----------------------------------------------
def compute_block_features(block, weights={'saliency':1.0, 'color':1.0, 'edge':1.0, 'texture':1.0}):
    # --- Saliency ---
    saliency_obj = cv2.saliency.StaticSaliencySpectralResidual_create()
    (success, saliencyMap) = saliency_obj.computeSaliency(block)
    saliencyMap = (saliencyMap*255).astype(np.float64)
    sal_avg, sal_std = saliencyMap.mean(), saliencyMap.std()
    sal_feat = np.array([sal_avg, sal_std]) * weights['saliency']

    # --- Color (HSV mean + std) ---
    hsv = cv2.cvtColor(block, cv2.COLOR_BGR2HSV).astype(np.float64)
    hsv_mean = hsv.mean(axis=(0,1))
    hsv_std = hsv.std(axis=(0,1))
    color_feat = np.concatenate([hsv_mean, hsv_std]) * weights['color']

    # --- Edge (Laplacian) ---
    gray = cv2.cvtColor(block, cv2.COLOR_BGR2GRAY)
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    edge_density = np.sum(np.abs(lap)>20)/lap.size
    edge_mean, edge_std = lap.mean(), lap.std()
    edge_feat = np.array([edge_density, edge_mean, edge_std]) * weights['edge']

    # --- Texture (LBP histogram) ---
    lbp = local_binary_pattern(gray, P=8, R=1, method='uniform')
    lbp_hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0,11), range=(0,10))
    lbp_hist = lbp_hist.astype(np.float64)
    lbp_hist /= (lbp_hist.sum() + 1e-6)
    texture_feat = lbp_hist * weights['texture']

    # Combine all
    features = np.concatenate([sal_feat, color_feat, edge_feat, texture_feat])
    return features

# ----------------------------------------------
# Extract features for sub-blocks
# ----------------------------------------------
def extract_features(img, grid_size=(8,8), sub_block=2, weights=None):
    if weights is None:
        weights = {'saliency':1.0, 'color':1.0, 'edge':1.0, 'texture':1.0}

    h, w = img.shape[:2]
    gh, gw = grid_size
    bh, bw = h//gh, w//gw

    features = []
    block_indices = []

    for i in range(gh):
        for j in range(gw):
            y0, y1 = i*bh, (i+1)*bh
            x0, x1 = j*bw, (j+1)*bw
            block = img[y0:y1, x0:x1]

            sub_feats = []
            sb_h, sb_w = bh//sub_block, bw//sub_block
            for si in range(sub_block):
                for sj in range(sub_block):
                    sy0, sy1 = si*sb_h, (si+1)*sb_h
                    sx0, sx1 = sj*sb_w, (sj+1)*sb_w
                    sub_block_img = block[sy0:sy1, sx0:sx1]
                    sub_feats.append(compute_block_features(sub_block_img, weights))

            features.extend(sub_feats)
            block_indices.append((i,j,len(sub_feats)))

    return np.array(features, dtype=np.float64), block_indices, (bh,bw)

# ----------------------------------------------
# Scale + PCA + KMeans clustering
# ----------------------------------------------
def cluster_features(features, n_clusters=2, pca_components=20,
                     weights={'saliency':2.0, 'color':1.0, 'edge':2.0, 'texture':1.0}):
    # --- 1️⃣ Scale features ---
    scaler = StandardScaler()
    feats_scaled = scaler.fit_transform(features)

    # --- 2️⃣ Apply feature weights proportionally ---
    # Assume feature order = [sal_avg, sal_std, hsv_mean(3), hsv_std(3),
    #                         edge_density, edge_mean, edge_std, lbp_hist(10)]
    # So total length = 2 + 6 + 3 + 10 = 21 features
    weight_vector = np.array(
        [weights['saliency']] * 2 +
        [weights['color']] * 6 +
        [weights['edge']] * 3 +
        [weights['texture']] * 10
    )

    # Normalize weights so total influence remains balanced
    weight_vector = weight_vector / np.mean(list(weights.values()))
    feats_weighted = feats_scaled * weight_vector

    # --- 3️⃣ PCA for dimensionality reduction ---
    pca = PCA(n_components=min(pca_components, feats_weighted.shape[1], feats_weighted.shape[0] - 1))
    feats_pca = pca.fit_transform(feats_weighted)

    # --- 4️⃣ GMM clustering ---
    gmm = GaussianMixture(n_components=n_clusters, random_state=42, covariance_type='full')
    labels = gmm.fit_predict(feats_pca)

    return labels

# ----------------------------------------------
# Aggregate sub-block labels to block
# ----------------------------------------------
def aggregate_majority(block_indices, sub_labels):
    gh = max(idx[0] for idx in block_indices)+1
    gw = max(idx[1] for idx in block_indices)+1
    block_labels = np.zeros((gh,gw),dtype=int)
    idx=0
    for i,j,sub_count in block_indices:
        sub_block_labels = sub_labels[idx:idx+sub_count]
        counts = np.bincount(sub_block_labels)
        block_labels[i,j] = np.argmax(counts)
        idx += sub_count
    return block_labels

# ----------------------------------------------
# Visualization
# ----------------------------------------------
def visualize_blocks(img_list, labels_list, block_shapes, cols=3):
    """
    Visualize block-level labels for multiple images.
    Each label value gets a distinct color overlay.
    Blocks are separated by thin white lines.
    """
    # --- Define distinct colors for up to 10 classes ---
    cmap = np.array([
        [1.0, 0.0, 0.0],   # red
        [0.0, 1.0, 0.0],   # green
        [0.0, 0.0, 1.0],   # blue
        [1.0, 1.0, 0.0],   # yellow
        [1.0, 0.0, 1.0],   # magenta
        [0.0, 1.0, 1.0],   # cyan
        [1.0, 0.5, 0.0],   # orange
        [0.5, 0.0, 1.0],   # purple
        [0.3, 0.3, 0.3],   # gray
        [1.0, 1.0, 1.0],   # white
    ])

    n_images = len(img_list)
    rows = (n_images + cols - 1) // cols
    plt.figure(figsize=(18, 6 * rows))

    for idx in range(n_images):
        img = img_list[idx].copy().astype(np.float64) / 255.0
        labels = labels_list[idx]
        bh, bw = block_shapes[idx]
        gh, gw = labels.shape

        overlay = img.copy()

        # --- Draw colored overlays per label ---
        for i in range(gh):
            for j in range(gw):
                label = int(labels[i, j])
                if label >= 0:  # skip -1 or invalid if any
                    color = cmap[label % len(cmap)]
                    y0, y1 = i * bh, (i + 1) * bh
                    x0, x1 = j * bw, (j + 1) * bw
                    overlay[y0:y1, x0:x1] = (
                        0.6 * overlay[y0:y1, x0:x1] + 0.4 * color
                    )

                    # --- Draw white block boundary ---
                    overlay[y0:y1, x0:x0+1] = [1, 1, 1]   # left line
                    overlay[y0:y1, x1-1:x1] = [1, 1, 1]   # right line
                    overlay[y0:y0+1, x0:x1] = [1, 1, 1]   # top line
                    overlay[y1-1:y1, x0:x1] = [1, 1, 1]   # bottom line

        plt.subplot(rows, cols, idx + 1)
        plt.imshow(cv2.cvtColor((overlay * 255).astype(np.uint8), cv2.COLOR_BGR2RGB))
        plt.axis("off")
        plt.title(f"Image {idx + 1}")

    plt.tight_layout()
    plt.show()

# ----------------------------------------------
# Full pipeline
# ----------------------------------------------
def process_folder_combined_features(folder_path, n_images=15, sub_block=2, weights=None):
    img_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg','.png','.jpeg'))]
    img_files = sorted(img_files)[:n_images]
    img_list, labels_list, block_shapes = [], [], []

    for img_file in img_files:
        img_path = os.path.join(folder_path, img_file)
        print(f"Processing {img_file}")
        img = cv2.imread(img_path)
        features, block_indices, block_shape = extract_features(img, sub_block=sub_block, weights=weights)
        sub_labels = cluster_features(features,2,20,weights)
        block_labels = aggregate_majority(block_indices, sub_labels)
        img_list.append(img)
        labels_list.append(block_labels)
        block_shapes.append(block_shape)

    visualize_blocks(img_list, labels_list, block_shapes)
    return img_list, labels_list, block_shapes

# ----------------------------------------------
# Example usage
# ----------------------------------------------
folder_path = r"F:\desktop\DS203\project\images_transformed"
weights = {'saliency':2.0, 'color':1.0, 'edge':2.0, 'texture':1.0}  # edge & saliency are weighted more
images, labels, shapes = process_folder_combined_features(folder_path, n_images=15, sub_block=4, weights=weights)


In [ ]:
import cv2
import numpy as np
import os
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from skimage.feature import local_binary_pattern

# ================================================================
# Compute combined features for a block
# ================================================================
def compute_block_features(block, weights={'saliency':1.0, 'color':1.0, 'edge':1.0, 'texture':1.0}):
    saliency_obj = cv2.saliency.StaticSaliencySpectralResidual_create()
    (success, saliencyMap) = saliency_obj.computeSaliency(block)
    saliencyMap = (saliencyMap*255).astype(np.float64)
    sal_avg, sal_std = saliencyMap.mean(), saliencyMap.std()
    sal_feat = np.array([sal_avg, sal_std]) * weights['saliency']

    hsv = cv2.cvtColor(block, cv2.COLOR_BGR2HSV).astype(np.float64)
    hsv_mean = hsv.mean(axis=(0,1))
    hsv_std = hsv.std(axis=(0,1))
    color_feat = np.concatenate([hsv_mean, hsv_std]) * weights['color']

    gray = cv2.cvtColor(block, cv2.COLOR_BGR2GRAY)
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    edge_density = np.sum(np.abs(lap)>20)/lap.size
    edge_mean, edge_std = lap.mean(), lap.std()
    edge_feat = np.array([edge_density, edge_mean, edge_std]) * weights['edge']

    lbp = local_binary_pattern(gray, P=8, R=1, method='uniform')
    lbp_hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0,11), range=(0,10))
    lbp_hist = lbp_hist.astype(np.float64)
    lbp_hist /= (lbp_hist.sum() + 1e-6)
    texture_feat = lbp_hist * weights['texture']

    return np.concatenate([sal_feat, color_feat, edge_feat, texture_feat])

# ================================================================
# Extract features for one image
# ================================================================
def extract_features(img, grid_size=(8,8), sub_block=2, weights=None):
    if weights is None:
        weights = {'saliency':1.0, 'color':1.0, 'edge':1.0, 'texture':1.0}

    h, w = img.shape[:2]
    gh, gw = grid_size
    bh, bw = h//gh, w//gw

    features = []
    block_indices = []

    for i in range(gh):
        for j in range(gw):
            y0, y1 = i*bh, (i+1)*bh
            x0, x1 = j*bw, (j+1)*bw
            block = img[y0:y1, x0:x1]

            sub_feats = []
            sb_h, sb_w = bh//sub_block, bw//sub_block
            for si in range(sub_block):
                for sj in range(sub_block):
                    sy0, sy1 = si*sb_h, (si+1)*sb_h
                    sx0, sx1 = sj*sb_w, (sj+1)*sb_w
                    sub_block_img = block[sy0:sy1, sx0:sx1]
                    sub_feats.append(compute_block_features(sub_block_img, weights))

            features.extend(sub_feats)
            block_indices.append((i,j,len(sub_feats)))

    return np.array(features, dtype=np.float64), block_indices, (bh,bw)

# ================================================================
# Global clustering
# ================================================================
def cluster_all_features(all_features, n_clusters=2, pca_components=20, weights=None):
    if weights is None:
        weights = {'saliency':1.0, 'color':1.0, 'edge':1.0, 'texture':1.0}

    scaler = StandardScaler()
    feats_scaled = scaler.fit_transform(all_features)

    weight_vector = np.array(
        [weights['saliency']]*2 +
        [weights['color']]*6 +
        [weights['edge']]*3 +
        [weights['texture']]*10
    )
    weight_vector = weight_vector / np.mean(list(weights.values()))
    feats_weighted = feats_scaled * weight_vector

    pca = PCA(n_components=min(pca_components, feats_weighted.shape[1], feats_weighted.shape[0]-1))
    feats_pca = pca.fit_transform(feats_weighted)

    gmm = GaussianMixture(n_components=n_clusters, random_state=42, covariance_type='full')
    labels = gmm.fit_predict(feats_pca)
    return labels

# ================================================================
# Aggregate sub-block labels to block level
# ================================================================
def aggregate_majority(block_indices, sub_labels):
    gh = max(idx[0] for idx in block_indices)+1
    gw = max(idx[1] for idx in block_indices)+1
    block_labels = np.zeros((gh,gw),dtype=int)
    idx=0
    for i,j,sub_count in block_indices:
        sub_block_labels = sub_labels[idx:idx+sub_count]
        counts = np.bincount(sub_block_labels)
        block_labels[i,j] = np.argmax(counts)
        idx += sub_count
    return block_labels

# ================================================================
# Save overlay with yellow blocks and numbers
# ================================================================
def save_overlay_image(img, block_labels, block_shape, save_path):
    overlay = img.copy().astype(np.float64)/255.0
    bh, bw = block_shape
    gh, gw = block_labels.shape
    yellow = np.array([0,1,1])  # BGR yellow
    font = cv2.FONT_HERSHEY_SIMPLEX

    for i in range(gh):
        for j in range(gw):
            block_id = i*gw + j
            y0, y1 = i*bh, (i+1)*bh
            x0, x1 = j*bw, (j+1)*bw
            overlay[y0:y1, x0:x1] = 0.5*overlay[y0:y1, x0:x1] + 0.5*yellow
            cv2.rectangle(overlay, (x0, y0), (x1-1, y1-1), (1,1,1), 1)
            cv2.putText(overlay, str(block_id), (x0+5, y0+20), font, 0.6, (0,0,0), 2, cv2.LINE_AA)

    out_img = (overlay*255).astype(np.uint8)
    cv2.imwrite(save_path, out_img)

# ================================================================
# Main Pipeline
# ================================================================
def process_folder_combined_features(folder_path, sub_block=2, weights=None, output_folder="images_cluster_gmm"):
    os.makedirs(output_folder, exist_ok=True)
    img_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg','.png','.jpeg'))]
    img_files = sorted(img_files)

    all_features = []
    image_feature_slices = []
    all_block_indices = []
    all_block_shapes = []
    all_images = []

    print("Extracting features from all images...")
    for img_file in img_files:
        img_path = os.path.join(folder_path, img_file)
        img = cv2.imread(img_path)
        feats, block_indices, block_shape = extract_features(img, sub_block=sub_block, weights=weights)
        start = len(all_features)
        all_features.extend(feats)
        end = len(all_features)
        image_feature_slices.append((start, end))
        all_block_indices.append(block_indices)
        all_block_shapes.append(block_shape)
        all_images.append((img_file, img))

    all_features = np.array(all_features, dtype=np.float64)
    print(f"Total feature vectors: {len(all_features)}")

    print("Performing global GMM clustering...")
    all_labels = cluster_all_features(all_features, n_clusters=2, pca_components=20, weights=weights)

    print("Aggregating and saving results...")
    rows = []
    for idx, (img_file, img) in enumerate(all_images):
        start, end = image_feature_slices[idx]
        sub_labels = all_labels[start:end]
        block_labels = aggregate_majority(all_block_indices[idx], sub_labels)

        save_path = os.path.join(output_folder, f"{os.path.splitext(img_file)[0]}_clustered.jpg")
        save_overlay_image(img, block_labels, all_block_shapes[idx], save_path)

        # --- prepare one row (filename + 64 clusters)
        row = [img_file] + block_labels.flatten().tolist()
        rows.append(row)

    # --- prepare CSV columns
    columns = ["filename"] + [f"block_{i}" for i in range(64)]
    df = pd.DataFrame(rows, columns=columns)
    df.to_csv(os.path.join(output_folder, "block_clusters.csv"), index=False)
    print(f"Saved clustered images and CSV to {output_folder}")

# ================================================================
# Example Run
# ================================================================
if __name__ == "__main__":
    folder_path = r"F:\desktop\DS203\project\images_transformed"
    weights = {'saliency':2.0, 'color':1.0, 'edge':2.0, 'texture':1.0}
    process_folder_combined_features(folder_path, sub_block=4, weights=weights)
